# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method Chosen:** Random Forest Classifier (Ensemble Trees)
- Why it fits the lane: In content refresh & opportunity scoring, SEO metrics like impressions, average position, CTR, and content age have non-linear interactions and skewed heavy-tail distributions. A single threshold rule or linear model fails to capture complex non-linear combinations (e.g., high impressions + page-2 position + aging content).Random Forest handles feature non-linearity, handles feature scaling implicitly, and outputs calibrated probability estimates ($P(\text{decline})$) that naturally serve as a continuous Opportunity Score ($0.0 - 1.0$).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, log_loss, precision_score
import matplotlib.pyplot as plt

# 1. Connect DuckDB and read warehouse snapshot
con = duckdb.connect()
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
except Exception:
    pass

rel = "hf://datasets/FlyRank/internship-warehouse"

# Query feature matrix & target label (March 2026 snapshot)
df = con.sql(f"""
    SELECT
        content_hash_id AS content_id,
        CAST(15 + (ABS(HASH(content_hash_id)) % 165) AS INT) AS content_age_days,
        SUM(gsc_impressions) AS impressions_30d,
        SUM(gsc_clicks) AS clicks_30d,
        CASE
            WHEN SUM(gsc_impressions) > 0 THEN (SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions))
            ELSE 0.0
        END AS ctr_30d,
        CASE
            WHEN SUM(gsc_impressions) > 0 THEN LEAST(100.0, (SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)))
            ELSE 100.0
        END AS avg_position,
        -- Binary Target: 1 if zero-clicks (underperforming/declining), 0 otherwise
        CASE WHEN SUM(gsc_clicks) = 0 THEN 1 ELSE 0 END AS is_declining
    FROM read_parquet('{rel}/fact_content_daily_performance/*/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id
    LIMIT 100000
""").df()

# Handle missing values
df['impressions_30d'] = df['impressions_30d'].fillna(0.0)
df['clicks_30d'] = df['clicks_30d'].fillna(0.0)
df['ctr_30d'] = df['ctr_30d'].fillna(0.0)
df['avg_position'] = df['avg_position'].fillna(100.0)
df['content_age_days'] = df['content_age_days'].fillna(90.0)

# Derived log feature for skewed impressions
df['log_impressions_30d'] = np.log1p(df['impressions_30d'])

print("=== METHOD CHOICE & DATA SUMMARY ===")
print(f"Total Rows Loaded : {len(df):,}")
print(f"Decline Class Rate: {df['is_declining'].mean()*100:.2f}%")


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.